In [ ]:
import sys
import torch
import matplotlib.pyplot as plt
import cv2
import os 

# Load Annotations

In [2]:
sys.path.append('/kaggle/input/notebooks/myahiia/dataset-annot-loader')

In [3]:
from utils.boxinfo import BoxInfo
annot_dict=torch.load('/kaggle/input/notebooks/myahiia/dataset-annot-loader/data/video_annot.pth',weights_only=False)

# All Labels

In [4]:
player_labels={}
clip_labels={}
for video_id,video in annot_dict.items():
    for clip_id, clip in video.items():
        if clip['category'] not in clip_labels:
            clip_labels[clip['category']]=1
        else:
            clip_labels[clip['category']]+=1
        for frame_id,frame in clip['frame_boxes_dct'].items():
            for player_box in frame:
                if player_box.category not in player_labels:
                    player_labels[player_box.category]=1
                else:
                    player_labels[player_box.category]+=1


print(player_labels)
print(clip_labels)

{'standing': 352368, 'setting': 11925, 'spiking': 11943, 'digging': 21393, 'waiting': 32823, 'blocking': 27162, 'moving': 46530, 'jumping': 3231, 'falling': 11187}
{'r_set': 644, 'r_spike': 623, 'l_set': 632, 'l-pass': 826, 'l-spike': 642, 'r-pass': 801, 'r_winpoint': 295, 'l_winpoint': 367}


In [5]:
player_labels = dict(sorted(player_labels.items(), key=lambda item: item[1]))
clip_labels = dict(sorted(clip_labels.items(), key=lambda item: item[1]))

In [6]:
for key,value in player_labels.items():
    print(key,value)

jumping 3231
falling 11187
setting 11925
spiking 11943
digging 21393
blocking 27162
waiting 32823
moving 46530
standing 352368


In [7]:
for key,value in clip_labels.items():
    print(key,value)

r_winpoint 295
l_winpoint 367
r_spike 623
l_set 632
l-spike 642
r_set 644
r-pass 801
l-pass 826


## Player Weighted labels

In [8]:
player_labels_weight={}
labels_sum=sum(player_labels.values())
num_classes=len(player_labels)

for label,count in player_labels.items():
    player_labels_weight[label]= round(labels_sum / (num_classes * count), 2)

print(player_labels_weight)

{'jumping': 17.83, 'falling': 5.15, 'setting': 4.83, 'spiking': 4.82, 'digging': 2.69, 'blocking': 2.12, 'waiting': 1.76, 'moving': 1.24, 'standing': 0.16}


# Only middle frame labels

In [15]:
player_labels={}
for video_id,video in annot_dict.items():
    for clip_id, clip in video.items():
       frame=list(clip['frame_boxes_dct'].values())[4]
       for box in frame:
            if box.category not in player_labels:
                player_labels[box.category]=1
            else:
                player_labels[box.category]+=1

In [16]:
player_labels

{'standing': 39152,
 'setting': 1325,
 'spiking': 1327,
 'digging': 2377,
 'waiting': 3647,
 'blocking': 3018,
 'moving': 5170,
 'jumping': 359,
 'falling': 1243}

## Player Weighted labels

In [17]:
player_labels_weight={}
labels_sum=sum(player_labels.values())
num_classes=len(player_labels)

for label,count in player_labels.items():
    player_labels_weight[label]= round(labels_sum / (num_classes * count), 2)

print(player_labels_weight)

{'standing': 0.16, 'setting': 4.83, 'spiking': 4.82, 'digging': 2.69, 'waiting': 1.76, 'blocking': 2.12, 'moving': 1.24, 'jumping': 17.83, 'falling': 5.15}


# Sample Frame Visualization

In [ ]:
def vis_frame(boxes_info, img_path):
    image = cv2.imread(img_path)

    # Create a figure with 3 rows and 4 columns
    fig, axes = plt.subplots(3, 4, figsize=(15, 10))
    axes = axes.flatten() # Flatten to 1D to iterate easily (0-11)

    for i, box_info in enumerate(boxes_info):        
        x1, y1, x2, y2 = box_info.box
        crop = image[y1:y2, x1:x2]
        
        axes[i].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[i].set_title(box_info.category)
        
        axes[i].axis('off') # Hide axes for all slots

    plt.tight_layout()
    plt.show()

vis_frame(annot_dict['25']['66355']['frame_boxes_dct'][66355],'/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos/25/66355/66355.jpg')

# Sample Clip Visualization

In [ ]:
def vis_clip(frame_boxes_dct, clip_dir):
    font = cv2.FONT_HERSHEY_SIMPLEX # text font 

    for frame_id, boxes_info in frame_boxes_dct.items():
        img_path = os.path.join(clip_dir, f'{frame_id}.jpg')
        image = cv2.imread(img_path)

        for box_info in boxes_info:
            x1, y1, x2, y2 = box_info.box
            #                                         colur     thickness
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, box_info.category, (x1, y1 - 10), font, 0.5, (0, 255, 0), 2)

        cv2.imshow('Image', image)
        cv2.waitKey(180)
    cv2.destroyAllWindows()

vis_clip(annot_dict['25']['66355']['frame_boxes_dct'],'/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos/25/66355')